# Example 10.2 — Full GA for Constrained Trajectory Synthesis

**Chapter 10: Complete Example: Step-by-Step Trajectory Synthesis**

Real-coded GA (nine design variables, BLX-0.3, tournament selection, elitism) with Grashof and transmission-angle penalties; 20 independent runs.

Only NumPy is used for the computation (`from numpy import *`); Matplotlib is used only for the figures.

In [1]:
import matplotlib.pyplot as plt
from numpy import *
from numpy import random

# ------------- Problem definition and GA parameters -------------
N_POINTS, N_POP, N_GEN = 20, 80, 300
PC, PM, SEED = 0.85, 0.04, 42
# x = [r1, r2, r3, r4, xp, yp, theta20 (deg), x0, y0]
LB = array([20, 20, 20, 20, -80, -80, 0, 0, 0], dtype=float)
UB = array([150, 150, 150, 150, 80, 80, 360, 200, 200],
           dtype=float)

def drop_trajectory(n=N_POINTS):
    t = linspace(0, 2 * pi, n, endpoint=False)
    f = 1 - 0.5 * sin(t / 2)
    return 100 + 50 * cos(t) * f, 100 + 50 * sin(t) * f

def analyze(r1, r2, r3, r4, t2):
    """Closed-form position analysis (one assembly branch)."""
    A = 2 * r4 * (r1 - r2 * cos(t2))
    B = -2 * r2 * r4 * sin(t2)
    C = r1**2 + r2**2 + r4**2 - r3**2 - 2 * r1 * r2 * cos(t2)
    disc = A**2 + B**2 - C**2
    root = sqrt(where(disc >= 0, disc, 0.0))
    t4 = where(disc >= 0, 2 * arctan2(-B - root, C - A), nan)
    t3 = arctan2(r4 * sin(t4) - r2 * sin(t2),
                 r1 + r4 * cos(t4) - r2 * cos(t2))
    return t3, t4

def transmission_angle(r1, r2, r3, r4, t2):
    """Angle between coupler and rocker, in degrees."""
    d2 = r1**2 + r2**2 - 2 * r1 * r2 * cos(t2)
    c = (r3**2 + r4**2 - d2) / (2 * r3 * r4)
    return degrees(arccos(clip(c, -1, 1)))

def coupler_point(x, t2, t3):
    r1, r2, r3, r4, xp, yp, t20, x0, y0 = x
    xa = x0 + r2 * cos(t2) + xp * cos(t3) - yp * sin(t3)
    ya = y0 + r2 * sin(t2) + xp * sin(t3) + yp * cos(t3)
    return xa, ya

def fitness(x, xd, yd):
    r1, r2, r3, r4, xp, yp, t20, x0, y0 = x
    L = sort(array([r1, r2, r3, r4]))
    # Grashof violation measure and crank condition
    g = max(0.0, L[0] + L[3] - L[1] - L[2]) \
        + max(0.0, r2 - L[0])
    PG = 1e4 + 100 * g if g > 0 else 0.0
    t2 = radians(t20) + linspace(0, 2 * pi, N_POINTS,
                                 endpoint=False)
    t3, t4 = analyze(r1, r2, r3, r4, t2)
    if isnan(t3).any():          # mechanism cannot be assembled
        return 1e5 + PG
    mu = transmission_angle(r1, r2, r3, r4, t2)
    Pmu = 50 * sum(maximum(0, 40 - mu) + maximum(0, mu - 140))
    xa, ya = coupler_point(x, t2, t3)
    return sum(sqrt((xa - xd)**2 + (ya - yd)**2)) + PG + Pmu

def genetic_algorithm(seed, verbose=False):
    random.seed(seed)
    xd, yd = drop_trajectory()
    pop = random.uniform(0, 1, (N_POP, 9)) * (UB - LB) + LB
    fit = array([fitness(p, xd, yd) for p in pop])
    best_h, mean_h, div_h = [], [], []
    for gen in range(N_GEN + 1):
        best_h.append(fit.min()); mean_h.append(fit.mean())
        div_h.append(mean(std(pop, axis=0) / (UB - LB)))
        if verbose and gen % 50 == 0:
            print(f'Gen {gen:>3}: best = {fit.min():7.2f}'
                  f' | mean = {fit.mean():9.2f}'
                  f' | div. = {div_h[-1]:.3f}')
        if gen == N_GEN:
            break
        # Tournament selection (size 2)
        i12 = random.randint(0, N_POP, (N_POP, 2))
        new = array([pop[a] if fit[a] < fit[b] else pop[b]
                     for a, b in i12])
        # BLX-0.3 crossover
        for i in range(0, N_POP - 1, 2):
            if random.rand() < PC:
                d = abs(new[i] - new[i + 1])
                lo = minimum(new[i], new[i + 1]) - 0.3 * d
                hi = maximum(new[i], new[i + 1]) + 0.3 * d
                new[i] = clip(lo + random.rand(9) * (hi - lo),
                              LB, UB)
                new[i + 1] = clip(lo + random.rand(9)
                                  * (hi - lo), LB, UB)
        # Gaussian mutation (sigma = 5% of the variable range)
        mask = random.rand(N_POP, 9) < PM
        new += mask * random.randn(N_POP, 9) * (UB - LB) * 0.05
        new = clip(new, LB, UB)
        new_fit = array([fitness(p, xd, yd) for p in new])
        # Elitism: the best individual always survives
        new[0], new_fit[0] = pop[argmin(fit)], fit.min()
        pop, fit = new, new_fit
    return (pop[argmin(fit)], array(best_h), array(mean_h),
            array(div_h))

# ----------------- Single run (seed 42) -----------------
xd, yd = drop_trajectory()
best, hbest, hmean, hdiv = genetic_algorithm(SEED, True)
names = ['r1', 'r2', 'r3', 'r4', 'xp', 'yp', 'theta20',
         'x0', 'y0']
units = ['mm'] * 6 + ['deg', 'mm', 'mm']
print('\nBest design found:')
for n_, v, u in zip(names, best, units):
    print(f'  {n_:>8} = {v:8.2f} {u}')

r1, r2, r3, r4 = best[:4]
t2 = radians(best[6]) + linspace(0, 2 * pi, N_POINTS,
                                 endpoint=False)
t3, t4 = analyze(r1, r2, r3, r4, t2)
xa, ya = coupler_point(best, t2, t3)
e = sqrt((xa - xd)**2 + (ya - yd)**2)
s, l = min(best[:4]), max(best[:4])
print(f'\nTotal error E = {e.sum():.2f} mm')
print(f'Error: mean = {e.mean():.2f} mm | '
      f'max = {e.max():.2f} mm | min = {e.min():.2f} mm')
print(f'Grashof: s + l = {s + l:.2f} < '
      f'p + q = {best[:4].sum() - s - l:.2f}')
print(f'Shortest link is the crank: {best[1] == s}')
conv = argmax(hbest <= 1.01 * hbest[-1])
print(f'Generation within 1% of final fitness: {conv}')

fig, ax = plt.subplots(1, 2, figsize=(6.4, 2.7))
ax[0].plot(xd, yd, 'ko-', ms=3, lw=0.8, label='Desired')
ax[0].plot(append(xa, xa[0]), append(ya, ya[0]), 's--',
           color='tab:red', ms=3, lw=0.8, label='GA design')
ax[0].set_aspect('equal')
ax[0].set_xlabel('x (mm)'); ax[0].set_ylabel('y (mm)')
ax[0].legend(fontsize=7)
ax[1].semilogy(hbest, 'k', lw=1.0, label='Best')
ax[1].semilogy(hmean, color='tab:blue', lw=0.8,
               label='Population mean')
ax[1].set_xlabel('Generation')
ax[1].set_ylabel('Fitness (mm)')
ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

# ------------ Repeatability: 20 independent runs ------------
finals, curves, divs, gens = [], [], [], []
for seed in range(20):
    _, hb, hm, hd = genetic_algorithm(seed)
    finals.append(hb[-1]); curves.append(hb); divs.append(hd)
    gens.append(argmax(hb <= 1.01 * hb[-1]))
finals = array(finals)
print(f'\n20 runs: best E = {finals.min():.2f} mm | '
      f'worst = {finals.max():.2f} mm')
print(f'         mean = {finals.mean():.2f} mm | '
      f'std = {finals.std():.2f} mm')
print(f'Generations within 1% of final fitness: '
      f'mean = {mean(gens):.0f}')
print(f'         min = {min(gens)}, max = {max(gens)}')
dm = mean(divs, axis=0)
print(f'Mean diversity: gen 0 = {dm[0]:.3f}, '
      f'gen 50 = {dm[50]:.3f}, gen 300 = {dm[300]:.3f}')

fig, ax = plt.subplots(1, 2, figsize=(6.4, 2.7))
for c in curves:
    ax[0].semilogy(c, color='0.7', lw=0.5)
ax[0].semilogy(median(curves, axis=0), 'k', lw=1.2,
               label='Median of 20 runs')
ax[0].semilogy(hbest, 'tab:red', lw=1.0, label=f'Seed {SEED}')
ax[0].set_xlabel('Generation')
ax[0].set_ylabel('Best fitness (mm)')
ax[0].legend(fontsize=7)
for d in divs:
    ax[1].plot(d, color='0.7', lw=0.5)
ax[1].plot(dm, 'k', lw=1.2, label='Mean of 20 runs')
ax[1].set_xlabel('Generation')
ax[1].set_ylabel('Design-space diversity')
ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

Gen   0: best =  818.99 | mean =  83523.10 | div. = 0.293
Gen  50: best =  108.18 | mean =    131.35 | div. = 0.016
Gen 100: best =  107.86 | mean =    130.63 | div. = 0.012
Gen 150: best =  107.63 | mean =    129.94 | div. = 0.008
Gen 200: best =  107.51 | mean =    153.63 | div. = 0.011
Gen 250: best =  107.21 | mean =    134.88 | div. = 0.010
Gen 300: best =  106.94 | mean =    149.96 | div. = 0.011

Best design found:
        r1 =   120.85 mm
        r2 =    32.53 mm
        r3 =   127.28 mm
        r4 =    69.83 mm
        xp =    -0.97 mm
        yp =     6.32 mm
   theta20 =     1.37 deg
        x0 =   108.62 mm
        y0 =    95.53 mm

Total error E = 106.94 mm
Error: mean = 5.35 mm | max = 13.02 mm | min = 2.64 mm
Grashof: s + l = 159.81 < p + q = 190.67
Shortest link is the crank: True
Generation within 1% of final fitness: 72

20 runs: best E = 106.10 mm | worst = 128.43 mm
         mean = 113.58 mm | std = 6.45 mm
Generations within 1% of final fitness: mean = 158
        